In [1]:
%cd /home/parthgandhi/Projects/SwingBot/swingbot/src

/home/parthgandhi/Projects/SwingBot/swingbot/src


In [3]:
import argparse
import logging
from datetime import datetime

import polars as pl

from computation.compute import gen_market_dashboard_data
from config.base import StorageConfig
from config.computation.compute import ComputeConfig
from config.ingestion.brokers import KiteConfig
from config.ingestion.data_sources import NSEConfig
from ingestion import fetch_nse_indices
from computation.scanner import prep_scan_data
from config.computation.indicator import IndicatorConfig

logger = logging.getLogger(__name__)

In [4]:
# get indices constituents
nse_indices_df = fetch_nse_indices(download_flag=False)

# get industry classification data
nse_ind_table_id = NSEConfig.get_db_tbl(
    tbl_name=NSEConfig.CLASSIFICATION_TABLE_ID, failed_tbl=False
)

query = f"""
select max(timestamp) from {nse_ind_table_id}
"""
max_timestamp = pl.read_database_uri(query=query, uri=NSEConfig.DB_CONN).item(0, 0)

logger.info(f"TimeStamp usef for NSE Industry Classification: {max_timestamp}")

query = f"""
select * from {nse_ind_table_id}
where timestamp = '{max_timestamp}'
"""

nse_ind_df = pl.read_database_uri(query=query, uri=NSEConfig.DB_CONN)

# Get Stock & Indices Data
stock_table_id = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=False
)
indices_table_id = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_INDICES, frequency="day", failed_tbl=False
)

query = f"""
    select *
    from {stock_table_id}
"""

stocks_df = pl.read_database_uri(query=query, uri=KiteConfig.DB_CONN)

query = f"""
    select *
    from {indices_table_id}
"""

indices_df = pl.read_database_uri(query=query, uri=KiteConfig.DB_CONN)

Failed for /home/parthgandhi/data/tmp/nse/indices/nse_index_nifty_reits_realty.csv. Error: found more fields than defined in 'Schema'

Consider setting 'truncate_ragged_lines=True'.


In [24]:
IndicatorConfig.LOOKBACK_DAYS_TO_MIN_RETURN_PCT

{1: 4.99, 5: 7.5, 21: 10, 63: 20, 126: 60}

In [22]:
scan_data = prep_scan_data(stocks_df).collect()

In [23]:
scan_data

symbol,timestamp,open,high,low,close,volume,close_sma_50,close_sma_200,close_ema_9,close_ema_21,volume_sma_20,volume_sma_50,day_range,body_by_range_pct_sma_20,body_by_range_pct_sma_50,lower_wick_by_range_pct_sma_20,lower_wick_by_range_pct_sma_50,upper_wick_by_range_pct_sma_20,upper_wick_by_range_pct_sma_50,adr_pct_20,rvol_pct_20,rvol_pct_50,clean_score_pct_20,clean_score_pct_50,close_prev_1,close_prev_5,close_prev_21,close_prev_63,close_prev_126,timestamp_prev_1,timestamp_prev_5,timestamp_prev_21,timestamp_prev_63,timestamp_prev_126,pct_gain_prev_1,pct_gain_prev_5,pct_gain_prev_21,pct_gain_prev_63,pct_gain_prev_126,all_data_flag
str,date,f64,f64,f64,f64,i64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,date,date,date,date,f64,f64,f64,f64,f64,bool
"""21STCENMGM""",2025-03-19,62.63,63.9,62.63,62.63,9760,null,null,62.63,62.63,null,null,1.0203,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
"""21STCENMGM""",2025-03-20,61.5,63.85,61.5,62.44,5768,null,null,62.52,62.53,null,null,1.0382,null,null,null,null,null,null,null,null,null,null,null,62.63,null,null,null,null,2025-03-19,null,null,null,null,-0.3034,null,null,null,null,false
"""21STCENMGM""",2025-03-21,62.44,63.68,62.35,63.5,16853,null,null,62.92,62.88,null,null,1.0213,null,null,null,null,null,null,null,null,null,null,null,62.44,null,null,null,null,2025-03-20,null,null,null,null,1.6976,null,null,null,null,false
"""21STCENMGM""",2025-03-24,64.77,64.77,64.77,64.77,1574,null,null,63.55,63.43,null,null,1.0,null,null,null,null,null,null,null,null,null,null,null,63.5,null,null,null,null,2025-03-21,null,null,null,null,2.0,null,null,null,null,false
"""21STCENMGM""",2025-03-25,66.06,66.06,66.06,66.06,4786,null,null,64.3,64.06,null,null,1.0,null,null,null,null,null,null,null,null,null,null,null,64.77,null,null,null,null,2025-03-24,null,null,null,null,1.9917,null,null,null,null,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""AARNAV""",2026-03-09,24.55,24.8,22.99,24.35,17335,null,null,25.63,25.94,null,null,1.0787,null,null,null,null,null,null,null,null,null,null,null,25.31,27.29,null,null,null,2026-03-06,2026-02-27,null,null,null,-3.793,-10.7732,null,null,null,false
"""AARNAV""",2026-03-10,24.36,25.1,23.5,24.09,19391,null,null,25.27,25.65,null,null,1.0681,null,null,null,null,null,null,null,null,null,null,null,24.35,26.67,null,null,null,2026-03-09,2026-03-02,null,null,null,-1.0678,-9.6738,null,null,null,false
"""AARNAV""",2026-03-11,24.4,28.3,24.01,26.29,30255,null,null,25.5,25.74,null,null,1.1787,null,null,null,null,null,null,null,null,null,null,null,24.09,25.16,null,null,null,2026-03-10,2026-03-04,null,null,null,9.1324,4.4913,null,null,null,false


In [33]:


scan_data.lazy().group_by(
    "timestamp",
).agg(
    [pl.col("symbol").count().alias("# Stocks")]
    + [
        (pl.col("close") >= pl.col(f"close_ema_{i}"))
        .sum()
        .alias(f"above_ema_{i}")
        for i in IndicatorConfig.EMA_DAYS
    ]
    + [
        (pl.col("close") >= pl.col(f"close_sma_{i}"))
        .sum()
        .alias(f"above_sma_{i}")
        for i in IndicatorConfig.SMA_DAYS
    ] + [
        (pl.col("pct_gain_prev_1") >= 4.5).sum().alias("UP 4.5% 1D"),
        (pl.col("pct_gain_prev_1") < 4.5).sum().alias("DOWN 4.5% 1D")
        
    ]
).with_columns(
    (pl.col(col) * 100 / pl.col("# Stocks")).round(2).alias(f"% {col.replace("_", " ").upper()}")
    for col in   ["UP 4.5% 1D", "DOWN 4.5% 1D" ] + [f"above_ema_{i}" for i in IndicatorConfig.EMA_DAYS] + [f"above_sma_{i}" for i in IndicatorConfig.SMA_DAYS] 
).select(
    pl.exclude(
        ["UP 4.5% 1D", "DOWN 4.5% 1D" ]+ [f"above_ema_{i}" for i in IndicatorConfig.EMA_DAYS] + [f"above_sma_{i}" for i in IndicatorConfig.SMA_DAYS]
    )
).sort("timestamp", descending=True).head(10).collect()

timestamp,# Stocks,% UP 4.5% 1D,% DOWN 4.5% 1D,% ABOVE EMA 9,% ABOVE EMA 21,% ABOVE SMA 50,% ABOVE SMA 200
date,u32,f64,f64,f64,f64,f64,f64
2026-03-13,2431,1.03,98.93,11.27,10.7,13.78,14.44
2026-03-12,2430,2.96,96.95,26.46,19.84,19.51,16.71
2026-03-11,2429,3.79,96.17,26.72,20.05,19.47,17.0
2026-03-10,2428,12.36,87.64,26.32,19.6,20.43,17.13
2026-03-09,2428,0.95,99.01,10.01,12.31,15.36,15.65
2026-03-06,2427,2.97,96.99,17.59,17.51,19.9,18.21
2026-03-05,2426,5.48,94.44,17.97,18.22,20.28,19.37
2026-03-04,2425,2.14,97.77,10.39,13.15,18.52,17.81
2026-03-02,2423,2.64,97.28,15.02,20.1,23.19,20.26
